# Real EEG round trip

Convert PhysioNet EEG Motor Movement/Imagery v1.0.0, subject S001 run 03. This is 64-channel scalp EEG with EDF+ task annotations. The source uses the Open Data Commons Attribution license. [Dataset and citation](https://doi.org/10.13026/C28G6P)

Install NeuroBridge with the `notebook` extra and use that Python environment as the kernel. The run uses a new local work directory so repeating it preserves previous outputs.

In [ ]:
from pathlib import Path
from tempfile import mkdtemp

from neurobridge.demo import fetch_demo
from neurobridge.edf import inspect_edf
from neurobridge.session import load_session

root = Path.cwd()
if root.name == "examples":
    root = root.parent
cache = root / ".local" / "physionet"
session = load_session(fetch_demo(cache))
recording, annotations = inspect_edf(session.recording)
print(recording.samples, "samples ×", len(recording.channels), "channels")
print(recording.rate_hz, "Hz,", len(annotations), "source intervals")
run = Path(mkdtemp(prefix="notebook-", dir=cache))
nwb = run / "demo.nwb"

## Convert and compare

SpikeInterface reads digital counts in windows. NeuroConv and PyNWB write the electrical series and annotations. The validation code separately decodes the source's physical values with PyEDFlib, reopens the NWB and compares all values and intervals. NWB Inspector reports the source's missing subject age, which this notebook explicitly accepts for local QC.

In [ ]:
from neurobridge.convert import convert
from neurobridge.report import write_report
from neurobridge.validation import validate

convert(session, nwb)
qc = validate(session, nwb, allow_missing_age=True)
assert qc.passed, qc.to_dict()
for finding in qc.science:
    print(finding.check, "·", finding.detail)
report = run / "qc.html"
write_report(session, nwb, qc, report)
print(report)

## Reopen and inspect

Plot a one-channel window and the source task intervals in the same EEG clock. For this run, T0 is rest, T1 is left fist, and T2 is right fist. The task codes are preserved verbatim in NWB.

In [ ]:
from io import BytesIO

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Image, display
from pynwb import NWBHDF5IO

from neurobridge.validation import voltage_block

with NWBHDF5IO(nwb, "r") as io:
    file = io.read()
    series = file.acquisition["NeuralVoltage"]
    frames = int(15 * series.rate)
    time = series.starting_time + np.arange(frames) / series.rate
    trace = voltage_block(series, 0, frames)[:, 0] * 1e6
    intervals = file.intervals["task_annotations"].to_dataframe()

fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(time, trace, lw=0.6, color="#125c70")
for row in intervals.itertuples():
    if row.start_time < time[-1]:
        stop = min(row.stop_time, time[-1])
        ax.axvspan(row.start_time, stop, color="#438a88", alpha=0.12)
        ax.text(
            (row.start_time + stop) / 2,
            0.97,
            row.label,
            ha="center",
            va="top",
            transform=ax.get_xaxis_transform(),
        )
ax.set(xlabel="Time from start (s)", ylabel=f"{recording.channels[0]} (µV)")
image = BytesIO()
fig.savefig(image, format="png", dpi=140, bbox_inches="tight")
display(Image(data=image.getvalue()))
plt.close(fig)